# Load Libraries and Setup Prerequisites

In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
!git clone https://github.com/MiguelPartosa/Thesis-FOS-BinaryClass-WSD.git
%cd Thesis-FOS-BinaryClass-WSD/Ensemble Model/

[WinError 3] The system cannot find the path specified: 'Thesis-FOS-BinaryClass-WSD/Ensemble Model/'
c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\Thesis-FOS-BinaryClass-WSD\Ensemble Model


c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\IPython\core\magics\osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


# Load Models

## Sentence Transformer -- Sentence Embeddings


In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('sentence-transformers/LaBSE')

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Processing

## Load Dataset

In [4]:
df_test = pd.read_excel('../../Dataset/Test_Set_WLabels.xlsx')
df_train= pd.read_excel('../../Dataset/Train_Set_WLabels.xlsx')
display(df_test.head(1),df_test.shape,df_train.head(1),df_train.shape)

,FOS,Word Sense,Verb,Usage,Is FOS
0,anak sa sala,usa ka anak nga dili-konsejero,['sala'],Ang anak sa sala nga si Maria kay ginaatiman g...,1


(322, 5)

,FOS,Word Sense,Verb,Usage,Is FOS
0,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Ang iyang kauban sa trabaho pirmi magpalapad o...,1


(1288, 5)

## Sentence Transformer

Removing the label column from the dataset for clarity

In [5]:
# df_train.drop(columns=['Is FOS'], inplace=True)
df = df_train.copy()

In [6]:
# df['Word Sense'] = df['Word Sense']
df.head(3)

,FOS,Word Sense,Verb,Usage,Is FOS
0,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Ang iyang kauban sa trabaho pirmi magpalapad o...,1
1,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Nagpalit siya og balayronon sa papel sa tindah...,0
2,murag namiya og tai,Usa ka tawo nga mibiya sa dili-iasa nga walay ...,[],"""Ang silingan murag namiya og tai, kalit lang ...",1


The embeddings are explicitly normalized so that 

### Generate Embeddings and Similarity Columns

> An alteration from the previously shown way of retrieving the embeddings are its normalization to work with PCA and containing all the new columns to be generated from embeddings and similarity score into one function.

In [17]:
def GetEmbeddings(words:str) -> np.ndarray:
    return embedding_model.encode(words, normalize_embeddings=True)

def ComputeSimilarity(embedding1,embedding2):
    return embedding_model.similarity(embedding1, embedding2)

In [41]:
def RetrieveEmbeddings(df) -> pd.DataFrame:
    df_embeddings = df.copy()
    df_embeddings['Sentence Embeddings'] = df['Word Sense'].apply(GetEmbeddings)
    df_embeddings['Verb Embeddings'] = df['Verb'].apply(GetEmbeddings)
    df_embeddings['Usage Embeddings'] = df['Usage'].apply(GetEmbeddings)
    df_embeddings['Similarity Scores'] = df_embeddings.apply(lambda row: ComputeSimilarity(row['Sentence Embeddings'], row['Usage Embeddings']), axis=1)
    return df_embeddings
df_embedded = RetrieveEmbeddings(df)
df_embedded.head(3)

,FOS,Word Sense,Verb,Usage,Is FOS,Sentence Embeddings,Verb Embeddings,Usage Embeddings,Similarity Scores
0,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Ang iyang kauban sa trabaho pirmi magpalapad o...,1,"[0.004979501, -0.03455616, 0.044118445, -0.056...","[-0.028341336, -0.006864522, -0.008147446, -0....","[-0.027987016, -0.007430709, 0.023322027, 0.01...",[[tensor(0.4529)]]
1,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Nagpalit siya og balayronon sa papel sa tindah...,0,"[0.004979501, -0.03455616, 0.044118445, -0.056...","[-0.028341336, -0.006864522, -0.008147446, -0....","[-0.053077772, -0.042660423, -0.037503656, -0....",[[tensor(0.3356)]]
2,murag namiya og tai,Usa ka tawo nga mibiya sa dili-iasa nga walay ...,[],"""Ang silingan murag namiya og tai, kalit lang ...",1,"[-0.03320765, -0.006950153, 0.012191928, 0.016...","[-0.02244177, 0.010921938, -0.016191624, -0.04...","[-0.01381951, 0.021733848, -0.01817845, 0.0346...",[[tensor(0.3334)]]


### Checkpoint for dataset
- Save

In [44]:
df_embedded.to_excel('../../Dataset/Ensemble/train_with_embeddings.xlsx',index=False)
df_embedded = pd.read_excel('../../Dataset/Ensemble/train_with_embeddings.xlsx')
df_embedded.head(1)

,FOS,Word Sense,Verb,Usage,Is FOS,Sentence Embeddings,Verb Embeddings,Usage Embeddings,Similarity Scores
0,magpalapad og papel,aron magbaton ug dungog sa buhat sa uban,['magpalapad'],Ang iyang kauban sa trabaho pirmi magpalapad o...,1,[ 0.0049795 -0.03455616 0.04411845 -0.056273...,[-2.83413362e-02 -6.86452212e-03 -8.14744644e-...,[-2.79870164e-02 -7.43070897e-03 2.33220272e-...,tensor([[0.4529]])


## PCA-Guided K-means


To address the comment made by Engr. Andrei Martin P. Diamante during the 50% implementation defense, a correlation matrix will show the necessity of  include a correlation of the columns from the embeddings

## Classification Models

# Output